In [3]:
!apt-get install openjdk-11-jdk-headless -qq

!wget https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

!tar -xzf spark-3.5.1-bin-hadoop3.tgz

!pip install findspark

--2026-06-28 05:51:43--  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz’

spark-3.5.1-bin-had 100%[===================>] 381.90M   802KB/s    in 6m 56s  

2026-06-28 05:58:40 (940 KB/s) - ‘spark-3.5.1-bin-hadoop3.tgz’ saved [400446614/400446614]



In [4]:
import os
import findspark

os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Assignment") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.1


In [5]:
import pandas as pd

data = {
    "product_id":[101,102,103,104],
    "category":["Electronics","Furniture","Electronics","Clothing"],
    "price":["1200","5000","3500","800"],
    "status":["Completed","Pending","Completed","Completed"],
    "amount":[1200,500,2500,900],
    "region":["North","South","East","North"],
    "priority":["Low","High","Low","High"],
    "base_price":[1000,4500,3000,700],
    "old_name":["A","B","C","D"],
    "user_id":[1,2,None,4]
}

pd.DataFrame(data).to_csv("source.csv",index=False)

In [7]:
df = spark.read.csv(
    "source.csv",
    header=True,
    inferSchema=True
)

In [8]:
from pyspark.sql.functions import col

df.filter(col("category")=="Electronics") \
.select("product_id","price") \
.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101| 1200|
|       103| 3500|
+----------+-----+



In [9]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("old_name","new_name")

df = df.withColumn(
    "price",
    col("price").cast("double")
)

In [11]:
df.show()

+----------+-----------+------+---------+------+------+--------+----------+--------+-------+
|product_id|   category| price|   status|amount|region|priority|base_price|new_name|user_id|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+
|       101|Electronics|1200.0|Completed|  1200| North|     Low|      1000|       A|    1.0|
|       102|  Furniture|5000.0|  Pending|   500| South|    High|      4500|       B|    2.0|
|       103|Electronics|3500.0|Completed|  2500|  East|     Low|      3000|       C|   NULL|
|       104|   Clothing| 800.0|Completed|   900| North|    High|       700|       D|    4.0|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+



In [12]:
from pyspark.sql.functions import col

df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
).show()

+----------+-----------+------+---------+------+------+--------+----------+--------+-------+
|product_id|   category| price|   status|amount|region|priority|base_price|new_name|user_id|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+
|       101|Electronics|1200.0|Completed|  1200| North|     Low|      1000|       A|    1.0|
|       103|Electronics|3500.0|Completed|  2500|  East|     Low|      3000|       C|   NULL|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+



In [14]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("base_price")*1.18
)

df.show()

+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|product_id|   category| price|   status|amount|region|priority|base_price|new_name|user_id|final_price|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|       101|Electronics|1200.0|Completed|  1200| North|     Low|      1000|       A|    1.0|     1180.0|
|       102|  Furniture|5000.0|  Pending|   500| South|    High|      4500|       B|    2.0|     5310.0|
|       103|Electronics|3500.0|Completed|  2500|  East|     Low|      3000|       C|   NULL|     3540.0|
|       104|   Clothing| 800.0|Completed|   900| North|    High|       700|       D|    4.0|      826.0|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+



In [17]:
df.write.mode("overwrite").parquet("input_parquet")

In [18]:
df = spark.read.parquet("input_parquet")

In [19]:
df.filter(df.user_id.isNotNull()) \
  .write.mode("overwrite") \
  .option("header", True) \
  .csv("output_csv")

In [21]:
df.show()

+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|product_id|   category| price|   status|amount|region|priority|base_price|new_name|user_id|final_price|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|       101|Electronics|1200.0|Completed|  1200| North|     Low|      1000|       A|    1.0|     1180.0|
|       102|  Furniture|5000.0|  Pending|   500| South|    High|      4500|       B|    2.0|     5310.0|
|       103|Electronics|3500.0|Completed|  2500|  East|     Low|      3000|       C|   NULL|     3540.0|
|       104|   Clothing| 800.0|Completed|   900| North|    High|       700|       D|    4.0|      826.0|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+



In [22]:
from pyspark.sql.functions import col

df.filter(
    (col("region")=="North") |
    (col("priority")=="High")
).show()

+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|product_id|   category| price|   status|amount|region|priority|base_price|new_name|user_id|final_price|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+
|       101|Electronics|1200.0|Completed|  1200| North|     Low|      1000|       A|    1.0|     1180.0|
|       102|  Furniture|5000.0|  Pending|   500| South|    High|      4500|       B|    2.0|     5310.0|
|       104|   Clothing| 800.0|Completed|   900| North|    High|       700|       D|    4.0|      826.0|
+----------+-----------+------+---------+------+------+--------+----------+--------+-------+-----------+

